# Task 4 — Uncertainty-budget audit

**Revision note (v2.0):** The Supplemental-Material uncertainty table in the submitted version contained entries that could not be reproduced from the code:

| Old entry | Actual origin | Problem |
|---|---|---|
| Finite sample & serial correlation 1.69×10⁻¹⁰ | mean of three HAC (lag=3) standard errors | statistically meaningless |
| Temperature sensor calibration 1.42×10⁻¹⁰ | EIV std with input 0.5 °C | spec is ±1.0 °C (2× understated) |
| Humidity sensor calibration 1.25×10⁻¹⁰ | EIV std with input 3.0 % | wrong magnitude |
| Pressure sensor calibration 9.11×10⁻¹³ | EIV std with input 12 Pa = 0.12 hPa | spec is ±1 hPa (8.3× understated) |

This notebook rebuilds the budget from first principles:

1. **Statistical uncertainties:** moving-block bootstrap (L = 156, B = 5000), cross-checked against HAC with lag = 156 (lag = 3 underestimates by ≈5×).
2. **Systematic uncertainties:** sensor *gain* errors from BME280 specifications (constant offsets are absorbed by the intercept $n_0$ and do not affect the coefficients).
3. **Two-measurand structure:** Table S1a (single-epoch $n$), Table S1b (coefficients).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
import sys, os, time
sys.path.append('..')
from models.mathar.Mathar2007 import n as n_mathar_scalar

df = pd.read_csv('../../data/processed/full_data.csv')
df = df.sort_values('time').reset_index(drop=True)

T_C, H_pct, P_hPa = df['temperature'].values, df['humidity'].values, df['pressure'].values
n_data = df['n_1762'].values

# Mathar cloud for residual decomposition (Table S1a)
lam_um, T_K, P_Pa = 1.762, T_C + 273.15, P_hPa * 100.0
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh) for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])

X = np.column_stack([np.ones(len(df)), T_C, H_pct, P_hPa])   # P in hPa -> coefficients in hPa^-1
N = len(df)
print(f"N = {N}")

N = 145784


In [2]:
# Statistical SE: moving-block bootstrap (L=156, B=5000)
def block_bootstrap_se(L=156, B=5000, seed=42):
    nb = N // L
    rng = np.random.default_rng(seed)
    W = np.zeros((nb, 4, 4)); U = np.zeros((nb, 4))
    for b in range(nb):
        i0 = b*L
        Xb = np.column_stack([np.ones(L), T_C[i0:i0+L], H_pct[i0:i0+L], P_hPa[i0:i0+L]])
        W[b] = Xb.T @ Xb; U[b] = Xb.T @ n_data[i0:i0+L]
    Wf = W.reshape(nb, 16)
    betas = np.empty((B, 4))
    for i in range(B):
        idx = rng.integers(0, nb, size=nb)
        XtX = Wf[idx].sum(axis=0).reshape(4,4)
        betas[i] = np.linalg.solve(XtX, U[idx].sum(axis=0))
    return betas

t0 = time.time()
boot = block_bootstrap_se()
print(f"Bootstrap done in {time.time()-t0:.0f} s")
se_boot = boot.std(axis=0)
names = ['n0','aT','aH','aP']
for j in range(1, 4):
    print(f"SE_boot[{names[j]}] = {se_boot[j]:.6e}")


# HAC (Newey-West) with lag = 156 (Bartlett), cross-check
model_ols = sm.OLS(n_data, X).fit()
hac156 = sm.OLS(n_data, X).fit(cov_type='HAC', cov_kwds={'maxlags': 156})
hac3   = sm.OLS(n_data, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
print()
print("Cross-check bootstrap vs HAC (lag=156):")
for j in range(1, 4):
    r = se_boot[j]/hac156.bse[j]
    print(f"  {names[j]}: boot={se_boot[j]:.6e}  HAC156={hac156.bse[j]:.6e}  ratio={r:.3f}")
print()
print("HAC lag=3 underestimation factors:")
for j in range(1, 4):
    print(f"  {names[j]}: HAC156/HAC3 = {hac156.bse[j]/hac3.bse[j]:.2f}x")

Bootstrap done in 0 s
SE_boot[aT] = 1.409194e-09
SE_boot[aH] = 1.146303e-09
SE_boot[aP] = 9.846054e-10

Cross-check bootstrap vs HAC (lag=156):
  aT: boot=1.409194e-09  HAC156=1.407200e-09  ratio=1.001
  aH: boot=1.146303e-09  HAC156=1.157710e-09  ratio=0.990
  aP: boot=9.846054e-10  HAC156=9.686824e-10  ratio=1.016

HAC lag=3 underestimation factors:
  aT: HAC156/HAC3 = 5.03x
  aH: HAC156/HAC3 = 5.16x
  aP: HAC156/HAC3 = 5.20x


In [3]:
# 3. Systematic uncertainties: BME280 sensor gain bounds (offset absorbed by n0; gain propagates as alpha_meas = alpha_true / g)
alpha_T = model_ols.params[1]
alpha_H = model_ols.params[2]
alpha_P = model_ols.params[3]          # hPa^-1 (design matrix uses hPa)

# campaign spans (observed)
span_T = (T_C.max() - T_C.min())
span_H = (H_pct.max() - H_pct.min())
span_P = (P_hPa.max() - P_hPa.min())
print(f"observed spans: T {span_T:.2f} K, H {span_H:.2f} %RH, P {span_P:.2f} hPa")

# worst-case gain error from manufacturer accuracy, rectangular -> /sqrt(3)
acc_T, acc_H, acc_P = 1.0, 3.0, 1.0   # degC, %RH (at 25 C), hPa

gain_err_T = 2*acc_T/span_T
gain_err_P = 2*acc_P/span_P

u_syst_T = (gain_err_T/np.sqrt(3)) * abs(alpha_T)
u_syst_P = (gain_err_P/np.sqrt(3)) * abs(alpha_P)
u_syst_H = 0.33 * abs(alpha_H)          # full worst-case band g in [1, 1.33], one-sided

print("Sensor gain systematics (1 sigma):")
print(f"  u_syst(alpha_T) = {u_syst_T:.3e}   (worst gain {100*gain_err_T:.1f}% / sqrt3)")
print(f"  u_syst(alpha_H) = {u_syst_H:.3e}   (g in [1,1.33], full band)")
print(f"  u_syst(alpha_P) = {u_syst_P:.3e}   (worst gain {100*gain_err_P:.1f}% / sqrt3)")

# combined
print()
print("Table S1b  —  Environmental coefficient uncertainty budget")
print("="*100)
print(f"{'Coeff':<14}{'Value':>16}{'u_stat':>14}{'u_syst':>14}{'u_tot':>14}{'U(k=2)':>14}{'HAC156':>14}")
vals = [
    ('alpha_T (K^-1)',  alpha_T, se_boot[1], u_syst_T, hac156.bse[1]),
    ('alpha_H (%RH^-1)', alpha_H, se_boot[2], u_syst_H, hac156.bse[2]),
    ('alpha_P (hPa^-1)', alpha_P, se_boot[3], u_syst_P, hac156.bse[3]),
]
for name, v, us, usy, uh in vals:
    ut = np.sqrt(us**2 + usy**2)
    print(f"{name:<14}{v:>16.6e}{us:>14.3e}{usy:>14.3e}{ut:>14.3e}{2*ut:>14.3e}{uh:>14.3e}")

observed spans: T 17.10 K, H 26.38 %RH, P 42.17 hPa
Sensor gain systematics (1 sigma):
  u_syst(alpha_T) = 5.973e-08   (worst gain 11.7% / sqrt3)
  u_syst(alpha_H) = 4.340e-09   (g in [1,1.33], full band)
  u_syst(alpha_P) = 7.106e-09   (worst gain 4.7% / sqrt3)

Table S1b  —  Environmental coefficient uncertainty budget
Coeff                    Value        u_stat        u_syst         u_tot        U(k=2)        HAC156
alpha_T (K^-1)   -8.847425e-07     1.409e-09     5.973e-08     5.974e-08     1.195e-07     1.407e-09
alpha_H (%RH^-1)   -1.315242e-08     1.146e-09     4.340e-09     4.489e-09     8.978e-09     1.158e-09
alpha_P (hPa^-1)    2.594906e-07     9.846e-10     7.106e-09     7.173e-09     1.435e-08     9.687e-10


In [4]:
# single-epoch budget (Table S1a) incl. residual decomposition
resid_data   = n_data - X @ model_ols.params
sigma_n      = np.std(resid_data, ddof=0)

# Mathar surrogate fit on identical rows -> model nonlinearity component
beta_m, *_ = np.linalg.lstsq(X, n_mathar, rcond=None)
sigma_model = np.std(n_mathar - X @ beta_m, ddof=0)
sigma_noise = np.sqrt(sigma_n**2 - sigma_model**2)

print(f"sigma_n (residual scatter)        = {sigma_n:.6e}")
print(f"  of which model nonlinearity     = {sigma_model:.6e}")
print(f"  of which measurement noise      = {sigma_noise:.6e}")
print(f"check: sqrt(noise^2+model^2)      = {np.sqrt(sigma_noise**2+sigma_model**2):.6e}")
print()
print("Table S1a  —  single-epoch refractive index budget")
print("="*70)
rows = [
    ("Residual scatter (incl. spatial/temporal mismatch)", sigma_n),
    ("780 nm frequency reference",                         1.0e-9),
    ("1762 nm frequency reference",                        0.0),   # <1e-12
]
tot = np.sqrt(sum(r[1]**2 for r in rows))
for name, v in rows:
    print(f"  {name:<55} {v:.3e}")
print(f"  {'Combined standard uncertainty':<55} {tot:.3e}")
print(f"  {'Combined expanded (k=2)':<55} {2*tot:.3e}")
print()
print("Residual decomposition (informational, not part of the RSS):")
print(f"  measurement noise  = {sigma_noise:.3e}")
print(f"  model nonlinearity = {sigma_model:.3e}")
print(f"  check sqrt(n^2+m^2)= {np.sqrt(sigma_noise**2+sigma_model**2):.3e}")

# significance of data-Mathar differences after systematics
print()
print("Significance of Delta-alpha after including sensor gain systematics:")
diffs = {
    'aT': (model_ols.params[1] - beta_m[1], se_boot[1], u_syst_T),
    'aH': (model_ols.params[2] - beta_m[2], se_boot[2], u_syst_H),
    'aP': (model_ols.params[3] - beta_m[3], se_boot[3], u_syst_P),
}
for k, (d, us, usy) in diffs.items():
    ut = np.sqrt(us**2 + usy**2)
    print(f"  Delta_{k} = {d:+.4e}  u_stat={us:.2e}  u_syst={usy:.2e}  -> covered by systematics: {abs(d) < 2*ut}")

sigma_n (residual scatter)        = 1.836658e-07
  of which model nonlinearity     = 4.430796e-08
  of which measurement noise      = 1.782412e-07
check: sqrt(noise^2+model^2)      = 1.836658e-07

Table S1a  —  single-epoch refractive index budget
  Residual scatter (incl. spatial/temporal mismatch)      1.837e-07
  780 nm frequency reference                              1.000e-09
  1762 nm frequency reference                             0.000e+00
  Combined standard uncertainty                           1.837e-07
  Combined expanded (k=2)                                 3.673e-07

Residual decomposition (informational, not part of the RSS):
  measurement noise  = 1.782e-07
  model nonlinearity = 4.431e-08
  check sqrt(n^2+m^2)= 1.837e-07

Significance of Delta-alpha after including sensor gain systematics:
  Delta_aT = -3.1062e-09  u_stat=1.41e-09  u_syst=5.97e-08  -> covered by systematics: True
  Delta_aH = +4.3334e-09  u_stat=1.15e-09  u_syst=4.34e-09  -> covered by systematics: Tr